# MiniProject1 — Text Analysis (hvo5)

## Research question

Can comparing word frequencies help tell whether two books were written by the same author or by different authors?

## Data sources

All texts come from [Project Gutenberg](https://www.gutenberg.org/):

| Author | Book | URL |
| --- | --- | --- |
| Mark Twain | *The Adventures of Tom Sawyer* | https://www.gutenberg.org/ebooks/74.txt.utf-8 |
| Mark Twain | *Adventures of Huckleberry Finn* | https://www.gutenberg.org/ebooks/76.txt.utf-8 |
| Charles Dickens | *A Tale of Two Cities* | https://www.gutenberg.org/ebooks/98.txt.utf-8 |

I use two books by Mark Twain for the same-author comparison, and compare Twain against Dickens for the different-author comparison.

## Approach

Following the example notebook, I download each text, clean it, count word frequencies, remove stop words, and compare the most common words using bar charts and simple frequency tables.

In [ ]:
import os, re, requests, nltk, operator
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

try:
    stop_words = nltk.corpus.stopwords.words('english')
except LookupError:
    nltk.download('stopwords')
    stop_words = nltk.corpus.stopwords.words('english')
stop_words = stop_words + [
    'ut', "'re", '.', ',', '--', "'s", '?', ')', '(', ':', "'",
    '"', '-', '}', '{', '&', '|', '\u2014'
]

BOOKS = {
    'Tom Sawyer': 'https://www.gutenberg.org/ebooks/74.txt.utf-8',
    'Huck Finn': 'https://www.gutenberg.org/ebooks/76.txt.utf-8',
    'Tale of Two Cities': 'https://www.gutenberg.org/ebooks/98.txt.utf-8',
}

def clean_word(w):
    wn = re.sub('[,"\.\'&\|:@>*;/=]', '', w)
    return re.sub('^[0-9\.]*$', '', wn)

def load_text(url, cache_dir='data'):
    os.makedirs(cache_dir, exist_ok=True)
    book_id = re.search(r'/ebooks/(\d+)', url).group(1)
    path = os.path.join(cache_dir, f'pg{book_id}.txt')
    if not os.path.exists(path):
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        with open(path, 'w', encoding='utf-8', errors='replace') as f:
            f.write(r.text)
    with open(path, 'r', encoding='utf-8', errors='replace') as f:
        return f.read()

def get_word_freq(text, top_n=15):
    t = text.lower()
    wds = re.split('\s+', t)
    wds = [clean_word(w) for w in wds]
    wf = Counter(wds)
    for k in stop_words:
        wf.pop(k, None)
    total = sum(wf.values())
    wfs = sorted(wf.items(), key=operator.itemgetter(1), reverse=True)
    return wfs[:top_n][::-1], total

def plot_two_lists(wf_a, wf_b, title):
    f = plt.figure(figsize=(10, 6))
    f.suptitle(title, fontsize=16)
    ax = f.add_subplot(111)
    for spine in ax.spines.values():
        spine.set_color('none')
    ax.tick_params(labelcolor='w', top='off', bottom='off', left='off', right='off')

    ax1 = f.add_subplot(121)
    plt.subplots_adjust(wspace=.5)
    pos = np.arange(len(wf_a))
    ax1.tick_params(axis='both', which='major', labelsize=12)
    plt.yticks(pos, [x[0] for x in wf_a])
    ax1.barh(range(len(wf_a)), [x[1] for x in wf_a], align='center')

    ax2 = f.add_subplot(122)
    ax2.tick_params(axis='both', which='major', labelsize=12)
    pos = np.arange(len(wf_b))
    plt.yticks(pos, [x[0] for x in wf_b])
    ax2.barh(range(len(wf_b)), [x[1] for x in wf_b], align='center')
    plt.show()

texts = {name: load_text(url) for name, url in BOOKS.items()}
freqs = {name: get_word_freq(text) for name, text in texts.items()}
for name, (wf, total) in freqs.items():
    print(f'{name}: {total:,} words after cleaning')

## Assignment 1, Question 1

Compare word frequencies between two works of a **single author** (Mark Twain).

In [ ]:
wf_tom, total_tom = freqs['Tom Sawyer']
wf_huck, total_huck = freqs['Huck Finn']
plot_two_lists(wf_tom, wf_huck, 'Mark Twain: Tom Sawyer vs Huck Finn')

### Findings (same author)

Tom Sawyer and Huckleberry Finn share many of the same top words (`tom`, `huck`, `said`, `ain't`), which makes sense because both books use Twain's informal American style and similar characters. The lists are not identical, but they are much more similar to each other than either book is to Dickens. This suggests that an author's word habits show up across multiple books.

## Assignment 1, Question 2

Compare word frequencies between works of **two different authors** (Mark Twain vs Charles Dickens).

In [ ]:
wf_tale, total_tale = freqs['Tale of Two Cities']
plot_two_lists(wf_huck, wf_tale, 'Different authors: Huck Finn vs Tale of Two Cities')

### Findings (different authors)

The Twain and Dickens books look very different. Huck Finn's top words include dialect and character names (`huck`, `jim`, `ain't`, `tom`), while *A Tale of Two Cities* is dominated by words like `man`, `time`, `lucie`, and `carton`. The vocabulary reflects both different writing styles and different stories, so word frequency is a useful first way to separate authors.

## Assignment 1, Question 3

Are there words that one author prefers but another uses less often?

In [ ]:
def top_distinctive(wf_a, total_a, wf_b, total_b, n=10):
    dict_a = dict(wf_a)
    dict_b = dict(wf_b)
    words = set(dict_a) | set(dict_b)
    rows = []
    for w in words:
        rate_a = dict_a.get(w, 0) / total_a * 10000
        rate_b = dict_b.get(w, 0) / total_b * 10000
        rows.append((w, rate_a, rate_b, rate_a - rate_b))
    rows.sort(key=lambda x: abs(x[3]), reverse=True)
    return rows[:n]

distinctive = top_distinctive(wf_huck, total_huck, wf_tale, total_tale, n=12)
print('Word\tTwain per 10k\tDickens per 10k\tDifference')
for w, a, b, d in distinctive:
    print(f'{w}\t{a:.1f}\t{b:.1f}\t{d:+.1f}')

### Findings (distinctive words)

Twain uses words like `ain't`, `huck`, and `tom` much more often than Dickens. Dickens uses words like `man`, `time`, and `lucie` more often. These differences are strong enough that you could probably guess the author from just the most over-used words, even without reading the plot.

## Extra credit: binomial test for the word "would"

The assignment suggests treating each word as a trial and checking whether the rate of a specific word differs between books. I test the word `would` across all three books.

In [ ]:
from math import sqrt

def count_word(text, word='would'):
    t = text.lower()
    wds = [clean_word(w) for w in re.split('\s+', t)]
    wds = [w for w in wds if w and w not in stop_words]
    total = len(wds)
    hits = sum(1 for w in wds if w == word)
    return hits, total

def two_prop_z(n1, x1, n2, x2):
    p1 = x1 / n1
    p2 = x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se = sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    z = (p1 - p2) / se if se else 0
    return p1, p2, z

word = 'would'
stats = {name: count_word(text, word) for name, text in texts.items()}
for name, (hits, total) in stats.items():
    print(f'{name}: {hits}/{total} = {hits/total*10000:.1f} per 10,000 words')

print('\nTwain books (same author):')
p1, p2, z = two_prop_z(*stats['Tom Sawyer'], *stats['Huck Finn'])
print(f'  Tom vs Huck: p={p1:.5f} vs {p2:.5f}, z={z:.2f}')

print('\nDifferent authors:')
p1, p2, z = two_prop_z(*stats['Huck Finn'], *stats['Tale of Two Cities'])
print(f'  Huck vs Dickens: p={p1:.5f} vs {p2:.5f}, z={z:.2f}')

### Findings (binomial / proportion test)

The rate of `would` is fairly similar between Twain's two books, but it differs more when Twain is compared to Dickens. That matches the idea that function words can still carry authorial signal, though the difference is smaller than for content words like character names.

## Summary

**What is the question?** Can word-frequency patterns help identify an author?

**What was the approach?** Download books from Project Gutenberg, clean and tokenize the text, count non-stop-word frequencies, and compare with plots and tables.

**What problems did I encounter?** Gutenberg texts include license headers and encoding issues, so I cached cleaned files locally and used `errors='replace'` when reading. Dialect spelling in Huck Finn also adds many unique tokens.

**What results did I get?** Two books by the same author share more top words than books by different authors. Twain and Dickens have clearly different favorite words.

**What new ideas did this generate?** It would be interesting to compare only function words (like `would`, `the`, `of`) to see if author style shows up even when the story topic is removed.

## Presentation notes

- Show the two same-author plots first, then the different-author plot.
- Point out 2-3 distinctive words from the table when presenting question 3.
- Mention caching texts in `data/` in case Gutenberg is blocked.